# 🗄️ Semana 3-4: Consultas SQL Avanzadas y Modelado de Datos

**Curso:** Python, SQL, Ciencia de Datos y Análisis de Negocios  
**Duración estimada:** 2 semanas  
**Nivel:** Intermedio

---

## 📋 ¿Qué vas a aprender esta semana?

| Bloque | Temas |
|--------|-------|
| 🔗 SQL Avanzado | `JOIN` (INNER, LEFT, RIGHT), subconsultas, funciones de agregación |
| ⚡ Índices y Optimización | Crear índices, medir rendimiento, `EXPLAIN QUERY PLAN` |
| 🏗️ Modelado de Datos | Diagrama ER, esquemas eficientes, claves primarias y foráneas |
| 🔧 Normalización | 1NF, 2NF, 3NF — cómo eliminar redundancias |
| ✏️ Reescritura de Consultas | Dividir queries complejas, CTEs, optimización práctica |

---

> 💡 **Prerequisito:** Haber completado la Semana 1-2.  
> Necesitás saber: `CREATE TABLE`, `INSERT`, `SELECT`, `WHERE`, `ORDER BY`, `GROUP BY`.

---
## 🔗 Bloque 1: Consultas SQL Avanzadas — JOINs

### 📘 Teoría

Un `JOIN` combina filas de dos o más tablas basándose en una columna relacionada.

```
Tabla A: clientes          Tabla B: pedidos
┌────┬────────┐            ┌────┬────────────┬──────────┐
│ id │ nombre │            │ id │ cliente_id │  total   │
├────┼────────┤            ├────┼────────────┼──────────┤
│  1 │  Ana   │            │  1 │     1      │  150.00  │
│  2 │  Luis  │            │  2 │     1      │   80.00  │
│  3 │  Marta │            │  3 │     2      │  200.00  │
└────┴────────┘            └────┴────────────┴──────────┘
```

#### Tipos de JOIN

| JOIN | Resultado |
|------|-----------|
| `INNER JOIN` | Solo filas que tienen coincidencia en **ambas** tablas |
| `LEFT JOIN` | Todas las filas de la tabla izquierda + coincidencias de la derecha (NULL si no hay) |
| `RIGHT JOIN` | Todas las filas de la derecha + coincidencias de la izquierda |
| `FULL OUTER JOIN` | Todas las filas de ambas tablas |

```sql
-- INNER JOIN: clientes que tienen pedidos
SELECT c.nombre, p.total
FROM clientes c
INNER JOIN pedidos p ON c.id = p.cliente_id;

-- LEFT JOIN: TODOS los clientes (aunque no tengan pedidos)
SELECT c.nombre, p.total
FROM clientes c
LEFT JOIN pedidos p ON c.id = p.cliente_id;
```

#### Funciones de Agregación
```sql
COUNT(*)          -- cantidad de filas
SUM(columna)      -- suma
AVG(columna)      -- promedio
MAX(columna)      -- máximo
MIN(columna)      -- mínimo
```

Siempre van acompañadas de `GROUP BY` cuando querés agrupar por categoría:
```sql
SELECT cliente_id, COUNT(*) as pedidos, SUM(total) as total_gastado
FROM pedidos
GROUP BY cliente_id
HAVING SUM(total) > 100;  -- HAVING filtra DESPUÉS de agrupar (WHERE filtra ANTES)
```

### 💡 Ejemplo resuelto 1 — Sistema de biblioteca con JOINs

In [ ]:
import sqlite3

# ✅ Base de datos de biblioteca con JOINs

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Crear tablas
cursor.executescript("""
    CREATE TABLE clientes (
        id      INTEGER PRIMARY KEY AUTOINCREMENT,
        nombre  TEXT    NOT NULL,
        email   TEXT    UNIQUE
    );

    CREATE TABLE libros (
        id      INTEGER PRIMARY KEY AUTOINCREMENT,
        titulo  TEXT    NOT NULL,
        autor   TEXT    NOT NULL,
        copias  INTEGER DEFAULT 1
    );

    CREATE TABLE prestamos (
        id          INTEGER PRIMARY KEY AUTOINCREMENT,
        cliente_id  INTEGER REFERENCES clientes(id),
        libro_id    INTEGER REFERENCES libros(id),
        fecha_prestamo  TEXT NOT NULL,
        fecha_devolucion TEXT
    );
""")

# Insertar datos
cursor.executemany('INSERT INTO clientes (nombre, email) VALUES (?, ?)', [
    ('Ana García',   'ana@mail.com'),
    ('Luis Pérez',   'luis@mail.com'),
    ('Marta López',  'marta@mail.com'),
    ('Carlos Ruiz',  'carlos@mail.com'),
])

cursor.executemany('INSERT INTO libros (titulo, autor, copias) VALUES (?, ?, ?)', [
    ('El Aleph',              'Borges',    3),
    ('Cien años de soledad',  'García Márquez', 2),
    ('Rayuela',               'Cortázar',  2),
    ('Ficciones',             'Borges',    1),
])

cursor.executemany(
    'INSERT INTO prestamos (cliente_id, libro_id, fecha_prestamo, fecha_devolucion) VALUES (?, ?, ?, ?)',
[
    (1, 1, '2024-01-05', '2024-01-15'),
    (1, 3, '2024-01-20', None),
    (2, 2, '2024-01-10', '2024-01-25'),
    (3, 1, '2024-02-01', '2024-02-10'),
    (3, 4, '2024-02-05', None),
])
conn.commit()

# ── INNER JOIN: préstamos con nombre de cliente y título
print('=== Préstamos activos (fecha_devolucion IS NULL) ===')
query = """
    SELECT c.nombre, l.titulo, p.fecha_prestamo
    FROM prestamos p
    INNER JOIN clientes c ON p.cliente_id = c.id
    INNER JOIN libros   l ON p.libro_id   = l.id
    WHERE p.fecha_devolucion IS NULL
    ORDER BY p.fecha_prestamo
"""
for row in cursor.execute(query):
    print(f'  {row[0]:15} | {row[1]:25} | desde {row[2]}')

# ── LEFT JOIN: TODOS los clientes, tengan o no préstamos activos
print('\n=== Clientes y sus préstamos activos (LEFT JOIN) ===')
query2 = """
    SELECT c.nombre,
           COALESCE(l.titulo, '— sin préstamo activo —') AS libro
    FROM clientes c
    LEFT JOIN prestamos p ON c.id = p.cliente_id AND p.fecha_devolucion IS NULL
    LEFT JOIN libros    l ON p.libro_id = l.id
    ORDER BY c.nombre
"""
for row in cursor.execute(query2):
    print(f'  {row[0]:15} | {row[1]}')

# ── Agregación: libros más prestados
print('\n=== Libros más prestados ===')
query3 = """
    SELECT l.titulo, COUNT(*) as veces_prestado
    FROM prestamos p
    INNER JOIN libros l ON p.libro_id = l.id
    GROUP BY l.titulo
    ORDER BY veces_prestado DESC
"""
for row in cursor.execute(query3):
    print(f'  {row[0]:25} | prestado {row[1]} vez/veces')

conn.close()

### ✏️ Ejercicio guiado 1 — Consultas con JOIN y filtros de fecha

Usando la misma BD de biblioteca, completá las consultas.

**Pistas:**
- Usá `INNER JOIN ... ON tabla_a.id = tabla_b.foreign_key`
- `HAVING COUNT(*) > 1` filtra grupos después de agrupar
- `strftime('%Y-%m', fecha)` extrae año-mes de una fecha en SQLite

In [ ]:
import sqlite3

# Recreamos la BD (ejecutá primero la celda del ejemplo)
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()
cursor.executescript("""
    CREATE TABLE clientes (id INTEGER PRIMARY KEY AUTOINCREMENT, nombre TEXT NOT NULL, email TEXT UNIQUE);
    CREATE TABLE libros (id INTEGER PRIMARY KEY AUTOINCREMENT, titulo TEXT NOT NULL, autor TEXT NOT NULL, copias INTEGER DEFAULT 1);
    CREATE TABLE prestamos (id INTEGER PRIMARY KEY AUTOINCREMENT, cliente_id INTEGER REFERENCES clientes(id), libro_id INTEGER REFERENCES libros(id), fecha_prestamo TEXT NOT NULL, fecha_devolucion TEXT);
""")
cursor.executemany('INSERT INTO clientes (nombre, email) VALUES (?, ?)', [('Ana García','ana@mail.com'),('Luis Pérez','luis@mail.com'),('Marta López','marta@mail.com'),('Carlos Ruiz','carlos@mail.com')])
cursor.executemany('INSERT INTO libros (titulo, autor, copias) VALUES (?, ?, ?)', [('El Aleph','Borges',3),('Cien años de soledad','García Márquez',2),('Rayuela','Cortázar',2),('Ficciones','Borges',1)])
cursor.executemany('INSERT INTO prestamos (cliente_id, libro_id, fecha_prestamo, fecha_devolucion) VALUES (?, ?, ?, ?)', [(1,1,'2024-01-05','2024-01-15'),(1,3,'2024-01-20',None),(2,2,'2024-01-10','2024-01-25'),(3,1,'2024-02-01','2024-02-10'),(3,4,'2024-02-05',None)])
conn.commit()

# ✏️ Consulta 1: libros prestados por Ana García en enero 2024
print('=== Préstamos de Ana García en enero 2024 ===')
# Pista: JOIN clientes + prestamos + libros, filtrar por nombre y mes
# cursor.execute("""SELECT ... WHERE c.nombre = 'Ana García' AND fecha_prestamo LIKE '2024-01%'""")


# ✏️ Consulta 2: clientes que tienen MÁS DE 1 préstamo en total
print('\n=== Clientes con más de 1 préstamo ===')
# Pista: GROUP BY c.nombre HAVING COUNT(*) > 1


# ✏️ Consulta 3: libros NUNCA prestados (LEFT JOIN + IS NULL)
print('\n=== Libros nunca prestados ===')
# Pista: LEFT JOIN prestamos ON l.id = p.libro_id WHERE p.id IS NULL


conn.close()

<details>
<summary>👀 Ver solución</summary>

```python
# Consulta 1
cursor.execute("""
    SELECT c.nombre, l.titulo, p.fecha_prestamo
    FROM prestamos p
    INNER JOIN clientes c ON p.cliente_id = c.id
    INNER JOIN libros   l ON p.libro_id   = l.id
    WHERE c.nombre = 'Ana García' AND p.fecha_prestamo LIKE '2024-01%'
""")

# Consulta 2
cursor.execute("""
    SELECT c.nombre, COUNT(*) as total_prestamos
    FROM prestamos p
    INNER JOIN clientes c ON p.cliente_id = c.id
    GROUP BY c.nombre
    HAVING COUNT(*) > 1
""")

# Consulta 3
cursor.execute("""
    SELECT l.titulo
    FROM libros l
    LEFT JOIN prestamos p ON l.id = p.libro_id
    WHERE p.id IS NULL
""")
```
</details>

### 🚀 Ejercicio libre 1 — Sistema de ventas

Creá una BD de e-commerce con estas tablas:
- `clientes`: id, nombre, ciudad
- `productos`: id, nombre, categoria, precio
- `pedidos`: id, cliente_id, fecha, estado ('pendiente'/'enviado'/'entregado')
- `detalle_pedido`: id, pedido_id, producto_id, cantidad

Insertá datos de prueba y respondé:
1. ¿Cuál es el **total gastado** por cada cliente? (`SUM(precio * cantidad)`)
2. ¿Qué productos **nunca fueron pedidos**? (LEFT JOIN + IS NULL)
3. ¿Cuáles son los **3 productos más vendidos** por cantidad? (TOP 3 con LIMIT)
4. ¿Qué clientes tienen pedidos en estado **'pendiente'**?

In [ ]:
import sqlite3

# 🚀 Tu solución aquí:


---
## ⚡ Bloque 2: Índices y Optimización de Consultas

### 📘 Teoría

Un **índice** es una estructura de datos separada que permite al motor de BD encontrar filas más rápido, sin escanear toda la tabla.

**Analogía:** el índice al final de un libro — en vez de leer todo, vas directo a la página.

```sql
-- Crear un índice en la columna 'email' de la tabla 'clientes'
CREATE INDEX idx_clientes_email ON clientes(email);

-- Índice compuesto (útil cuando filtrás por ambas columnas juntas)
CREATE INDEX idx_prestamos_fechas ON prestamos(cliente_id, fecha_prestamo);

-- Ver el plan de ejecución (¿usa índice o no?)
EXPLAIN QUERY PLAN SELECT * FROM clientes WHERE email = 'ana@mail.com';
```

#### ¿Cuándo usar índices?

| ✅ Sí conviene indexar | ❌ No conviene indexar |
|----------------------|----------------------|
| Columnas que usás en `WHERE` frecuentemente | Tablas muy pequeñas |
| Columnas usadas en `JOIN` | Columnas con muy pocos valores distintos |
| Columnas con `ORDER BY` | Tablas con muchas escrituras (el índice se mantiene) |
| Claves foráneas | |

#### Tradeoff
> Los índices **aceleran lecturas** pero **ralentizan escrituras** (INSERT/UPDATE/DELETE), porque el índice también debe actualizarse. Usalos con criterio.

### 💡 Ejemplo resuelto 2 — Comparar rendimiento con y sin índice

In [ ]:
import sqlite3
import time
import random

# ✅ Demostración de mejora de rendimiento con índices

def crear_bd_grande(con_indice=False):
    """Crea una BD con 100.000 registros, opcionalmente con índice."""
    conn = sqlite3.connect(':memory:')
    cursor = conn.cursor()

    cursor.execute("""
        CREATE TABLE empleados (
            id         INTEGER PRIMARY KEY AUTOINCREMENT,
            nombre     TEXT,
            email      TEXT,
            depto      TEXT,
            salario    REAL
        )
    """)

    deptos = ['Ventas', 'IT', 'RRHH', 'Marketing', 'Finanzas']
    datos = [
        (f'Empleado_{i}', f'emp{i}@empresa.com', random.choice(deptos), random.uniform(30000, 150000))
        for i in range(100_000)
    ]
    cursor.executemany(
        'INSERT INTO empleados (nombre, email, depto, salario) VALUES (?, ?, ?, ?)',
        datos
    )

    if con_indice:
        cursor.execute('CREATE INDEX idx_email  ON empleados(email)')
        cursor.execute('CREATE INDEX idx_depto  ON empleados(depto)')

    conn.commit()
    return conn


# Consulta de prueba
QUERY_EMAIL  = "SELECT * FROM empleados WHERE email = 'emp50000@empresa.com'"
QUERY_DEPTO  = "SELECT depto, COUNT(*), AVG(salario) FROM empleados GROUP BY depto"

# Sin índice
conn_sin = crear_bd_grande(con_indice=False)
t0 = time.perf_counter()
conn_sin.execute(QUERY_EMAIL).fetchall()
t_sin_email = time.perf_counter() - t0

t0 = time.perf_counter()
conn_sin.execute(QUERY_DEPTO).fetchall()
t_sin_depto = time.perf_counter() - t0
conn_sin.close()

# Con índice
conn_con = crear_bd_grande(con_indice=True)
t0 = time.perf_counter()
conn_con.execute(QUERY_EMAIL).fetchall()
t_con_email = time.perf_counter() - t0

t0 = time.perf_counter()
conn_con.execute(QUERY_DEPTO).fetchall()
t_con_depto = time.perf_counter() - t0
conn_con.close()

print('Consulta por email exacto (100k filas):')
print(f'  Sin índice: {t_sin_email*1000:.2f} ms')
print(f'  Con índice: {t_con_email*1000:.2f} ms')
mejora_email = t_sin_email / t_con_email if t_con_email > 0 else float('inf')
print(f'  Mejora:     {mejora_email:.1f}x más rápido')

print('\nConsulta GROUP BY departamento:')
print(f'  Sin índice: {t_sin_depto*1000:.2f} ms')
print(f'  Con índice: {t_con_depto*1000:.2f} ms')

# Mostrar plan de ejecución
conn_demo = crear_bd_grande(con_indice=True)
print('\n=== EXPLAIN QUERY PLAN — con índice ===')
for row in conn_demo.execute(f'EXPLAIN QUERY PLAN {QUERY_EMAIL}'):
    print(f'  {row}')
conn_demo.close()

### ✏️ Ejercicio guiado 2 — Medir y optimizar

Creá una BD de productos con 50.000 registros y comparé el tiempo de una consulta antes y después de agregar un índice.

**Pistas:**
- Usá `time.perf_counter()` para medir tiempos
- Probá con `WHERE categoria = ?` — ese es un buen candidato para índice
- Usá `EXPLAIN QUERY PLAN` para ver si SQLite usa el índice

In [ ]:
import sqlite3, time, random

categorias = ['Electrónica', 'Ropa', 'Hogar', 'Deportes', 'Libros']

def crear_productos(n=50_000, con_indice=False):
    conn = sqlite3.connect(':memory:')
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE productos (
            id        INTEGER PRIMARY KEY AUTOINCREMENT,
            nombre    TEXT,
            categoria TEXT,
            precio    REAL,
            stock     INTEGER
        )
    """)
    datos = [
        (f'Producto_{i}', random.choice(categorias),
         round(random.uniform(10, 5000), 2), random.randint(0, 500))
        for i in range(n)
    ]
    cursor.executemany('INSERT INTO productos (nombre, categoria, precio, stock) VALUES (?, ?, ?, ?)', datos)

    if con_indice:
        pass  # ✏️ Tu código aquí: agregá el índice en 'categoria'

    conn.commit()
    return conn


QUERY = "SELECT categoria, COUNT(*), AVG(precio) FROM productos WHERE categoria = 'Electrónica' GROUP BY categoria"

# ✏️ Medí el tiempo SIN índice
conn_sin = crear_productos(con_indice=False)
# Tu código aquí


# ✏️ Medí el tiempo CON índice
conn_con = crear_productos(con_indice=True)
# Tu código aquí


# ✏️ Mostrá la mejora y el EXPLAIN QUERY PLAN


<details>
<summary>👀 Ver solución</summary>

```python
# En crear_productos, con con_indice=True:
cursor.execute('CREATE INDEX idx_categoria ON productos(categoria)')

# Medir tiempos:
t0 = time.perf_counter()
conn_sin.execute(QUERY).fetchall()
t_sin = time.perf_counter() - t0

t0 = time.perf_counter()
conn_con.execute(QUERY).fetchall()
t_con = time.perf_counter() - t0

print(f'Sin índice: {t_sin*1000:.2f} ms')
print(f'Con índice: {t_con*1000:.2f} ms')
print(f'Mejora: {t_sin/t_con:.1f}x')

for row in conn_con.execute(f'EXPLAIN QUERY PLAN {QUERY}'):
    print(row)
```
</details>

### 🚀 Ejercicio libre 2 — Benchmark completo

Creá una BD de logs de acceso web con 200.000 registros con columnas: `ip`, `url`, `status_code`, `timestamp`, `tiempo_respuesta_ms`.

1. Medí el tiempo de estas 3 consultas **sin ningún índice**:
   - Buscar todos los logs de una IP específica
   - Contar accesos por `status_code`
   - URLs con tiempo de respuesta > 500ms
2. Agregá los índices que consideres necesarios
3. Volvé a medir y mostrá la tabla comparativa de resultados
4. Comentá cuál mejora más y por qué

In [ ]:
import sqlite3, time, random

# 🚀 Tu solución aquí:


---
## 🏗️ Bloque 3: Modelado de Datos — Esquemas y ER

### 📘 Teoría

El **modelado de datos** es el proceso de diseñar cómo se va a estructurar la información en una base de datos antes de implementarla.

#### Diagrama Entidad-Relación (ER)

Representa las **entidades** (cosas del mundo real), sus **atributos** y las **relaciones** entre ellas.

```
CLIENTE ──────────< PEDIDO >──────────── PRODUCTO
  - id (PK)           - id (PK)            - id (PK)
  - nombre            - cliente_id (FK)     - nombre
  - email             - producto_id (FK)    - precio
                      - cantidad
                      - fecha
```

#### Tipos de relaciones

| Relación | Descripción | Ejemplo |
|----------|-------------|----------|
| **1:1** | Un registro se relaciona con exactamente uno | persona → pasaporte |
| **1:N** | Un registro se relaciona con muchos | cliente → pedidos |
| **N:M** | Muchos con muchos (necesita tabla intermedia) | estudiantes ↔ cursos |

#### Implementación de relaciones N:M
```sql
-- La relación N:M se resuelve con una tabla intermedia
CREATE TABLE estudiantes (id INTEGER PRIMARY KEY, nombre TEXT);
CREATE TABLE cursos      (id INTEGER PRIMARY KEY, nombre TEXT);

-- Tabla intermedia = una fila por cada par (estudiante, curso)
CREATE TABLE inscripciones (
    estudiante_id INTEGER REFERENCES estudiantes(id),
    curso_id      INTEGER REFERENCES cursos(id),
    fecha_inscripcion TEXT,
    PRIMARY KEY (estudiante_id, curso_id)  -- clave primaria compuesta
);
```

### 💡 Ejemplo resuelto 3 — Tienda online: diseño e implementación

In [ ]:
import sqlite3

# ✅ Esquema completo de tienda online con todas las relaciones

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()
cursor.execute('PRAGMA foreign_keys = ON')  # habilitar FK en SQLite

cursor.executescript("""
    -- 1. Tablas maestras (sin dependencias)
    CREATE TABLE categorias (
        id     INTEGER PRIMARY KEY AUTOINCREMENT,
        nombre TEXT UNIQUE NOT NULL
    );

    CREATE TABLE clientes (
        id       INTEGER PRIMARY KEY AUTOINCREMENT,
        nombre   TEXT NOT NULL,
        email    TEXT UNIQUE NOT NULL,
        ciudad   TEXT
    );

    -- 2. Productos referencian a categorías (1:N)
    CREATE TABLE productos (
        id           INTEGER PRIMARY KEY AUTOINCREMENT,
        nombre       TEXT    NOT NULL,
        precio       REAL    NOT NULL CHECK(precio > 0),
        stock        INTEGER NOT NULL DEFAULT 0,
        categoria_id INTEGER REFERENCES categorias(id)
    );

    -- 3. Pedidos referencian a clientes (1:N)
    CREATE TABLE pedidos (
        id         INTEGER PRIMARY KEY AUTOINCREMENT,
        cliente_id INTEGER NOT NULL REFERENCES clientes(id),
        fecha      TEXT    NOT NULL DEFAULT (date('now')),
        estado     TEXT    NOT NULL DEFAULT 'pendiente'
                           CHECK(estado IN ('pendiente','enviado','entregado','cancelado'))
    );

    -- 4. Detalle_pedido: tabla intermedia pedidos <-> productos (N:M)
    CREATE TABLE detalle_pedido (
        pedido_id   INTEGER REFERENCES pedidos(id),
        producto_id INTEGER REFERENCES productos(id),
        cantidad    INTEGER NOT NULL CHECK(cantidad > 0),
        precio_unit REAL    NOT NULL,
        PRIMARY KEY (pedido_id, producto_id)
    );
""")

# Poblar con datos
cursor.executemany('INSERT INTO categorias (nombre) VALUES (?)', [('Electrónica',),('Ropa',),('Hogar',)])
cursor.executemany('INSERT INTO clientes (nombre, email, ciudad) VALUES (?, ?, ?)', [
    ('Ana García', 'ana@mail.com', 'Buenos Aires'),
    ('Luis Pérez', 'luis@mail.com', 'Córdoba'),
    ('Marta López', 'marta@mail.com', 'Rosario'),
])
cursor.executemany('INSERT INTO productos (nombre, precio, stock, categoria_id) VALUES (?, ?, ?, ?)', [
    ('Laptop Pro',  1200.00, 15, 1),
    ('Auriculares', 89.99,   50, 1),
    ('Camiseta',    25.00,  100, 2),
    ('Sillón',     350.00,   8, 3),
])
cursor.executemany('INSERT INTO pedidos (cliente_id, fecha, estado) VALUES (?, ?, ?)', [
    (1, '2024-01-10', 'entregado'),
    (1, '2024-02-05', 'enviado'),
    (2, '2024-01-20', 'entregado'),
])
cursor.executemany('INSERT INTO detalle_pedido (pedido_id, producto_id, cantidad, precio_unit) VALUES (?, ?, ?, ?)', [
    (1, 1, 1, 1200.00),
    (1, 2, 2, 89.99),
    (2, 3, 3, 25.00),
    (3, 4, 1, 350.00),
])
conn.commit()

# Consulta completa: total por pedido con datos del cliente
print('=== Resumen de pedidos ===')
query = """
    SELECT
        p.id          AS pedido,
        c.nombre      AS cliente,
        c.ciudad,
        p.fecha,
        p.estado,
        SUM(dp.cantidad * dp.precio_unit) AS total
    FROM pedidos p
    INNER JOIN clientes       c  ON p.cliente_id  = c.id
    INNER JOIN detalle_pedido dp ON p.id          = dp.pedido_id
    GROUP BY p.id
    ORDER BY p.fecha
"""
for row in cursor.execute(query):
    print(f'  Pedido #{row[0]} | {row[1]:12} ({row[2]}) | {row[3]} | {row[4]:10} | ${row[5]:.2f}')

# Total gastado por cliente
print('\n=== Total gastado por cliente ===')
query2 = """
    SELECT c.nombre, SUM(dp.cantidad * dp.precio_unit) as total
    FROM clientes c
    INNER JOIN pedidos p ON c.id = p.cliente_id
    INNER JOIN detalle_pedido dp ON p.id = dp.pedido_id
    GROUP BY c.nombre
    ORDER BY total DESC
"""
for row in cursor.execute(query2):
    print(f'  {row[0]:15} | ${row[1]:.2f}')

conn.close()

### ✏️ Ejercicio guiado 3 — Esquema de cine

Diseñá e implementá una BD para un sistema de cine con las siguientes entidades:

- `peliculas`: id, titulo, duracion_min, clasificacion
- `salas`: id, nombre, capacidad
- `funciones`: id, pelicula_id, sala_id, fecha_hora, precio
- `reservas`: id, funcion_id, nombre_cliente, cantidad_butacas

**Pistas:**
- `funciones` relaciona `peliculas` y `salas` (N:M resuelto con la tabla funciones)
- Usá `CHECK` para validar que `capacidad > 0` y `precio > 0`
- Insertá al menos 3 funciones y 4 reservas

In [ ]:
import sqlite3

# ✏️ Tu esquema de cine aquí:

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()
cursor.execute('PRAGMA foreign_keys = ON')

# Crear tablas:


# Insertar datos:


# Consulta: funciones disponibles con película y sala:


conn.close()


<details>
<summary>👀 Ver solución</summary>

```python
cursor.executescript("""
    CREATE TABLE peliculas (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        titulo TEXT NOT NULL, duracion_min INTEGER, clasificacion TEXT
    );
    CREATE TABLE salas (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        nombre TEXT NOT NULL, capacidad INTEGER CHECK(capacidad > 0)
    );
    CREATE TABLE funciones (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        pelicula_id INTEGER REFERENCES peliculas(id),
        sala_id INTEGER REFERENCES salas(id),
        fecha_hora TEXT NOT NULL,
        precio REAL CHECK(precio > 0)
    );
    CREATE TABLE reservas (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        funcion_id INTEGER REFERENCES funciones(id),
        nombre_cliente TEXT NOT NULL,
        cantidad_butacas INTEGER CHECK(cantidad_butacas > 0)
    );
""")
```
</details>

### 🚀 Ejercicio libre 3 — Diseñá tu propio esquema

Elegí uno de estos dominios y diseñá un esquema completo:

**Opción A:** Red social (usuarios, publicaciones, comentarios, likes, seguidores)

**Opción B:** Sistema hospitalario (pacientes, médicos, turnos, tratamientos, medicamentos)

**Opción C:** Plataforma de cursos online (instructores, cursos, módulos, estudiantes, progreso)

Requisitos:
1. Identificá todas las entidades y sus atributos
2. Definí los tipos de relaciones (1:1, 1:N, N:M)
3. Implementá con `sqlite3` con claves foráneas habilitadas
4. Insertá datos de prueba
5. Escribí al menos 3 consultas con JOIN que respondan preguntas de negocio

In [ ]:
import sqlite3

# 🚀 Tu diseño aquí:


---
## 🔧 Bloque 4: Normalización de Datos

### 📘 Teoría

La **normalización** es un proceso para organizar una base de datos eliminando redundancias y dependencias problemáticas.

#### El problema sin normalizar

```
Tabla VENTAS (desnormalizada — todo junto):
┌──────────────┬──────────┬─────────────────┬──────────────┬──────────┬────────┐
│ pedido_id    │ cliente  │ email_cliente    │ ciudad       │ producto │ precio │
├──────────────┼──────────┼─────────────────┼──────────────┼──────────┼────────┤
│ 1            │ Ana      │ ana@mail.com     │ Buenos Aires │ Laptop   │ 1200   │
│ 1            │ Ana      │ ana@mail.com     │ Buenos Aires │ Mouse    │  25    │  ← REDUNDANCIA
│ 2            │ Ana      │ ana@mail.com     │ Buenos Aires │ Teclado  │  80    │  ← REDUNDANCIA
└──────────────┴──────────┴─────────────────┴──────────────┴──────────┴────────┘
Problema: si Ana cambia su email, hay que actualizar 3 filas. Si olvidamos una → inconsistencia.
```

#### Las 3 Formas Normales

**1NF — Primera Forma Normal:**  
- Cada celda tiene un único valor atómico (sin listas ni grupos)
- Cada fila es única (tiene clave primaria)

**2NF — Segunda Forma Normal:**  
- Cumple 1NF
- Todos los atributos no-clave dependen de **toda** la clave primaria (no de una parte)

**3NF — Tercera Forma Normal:**  
- Cumple 2NF
- No hay **dependencias transitivas**: los atributos no-clave no dependen de otros atributos no-clave

> 📌 **Regla práctica:** si al cambiar un dato necesitás actualizar más de una fila, la tabla probablemente no está normalizada.

### 💡 Ejemplo resuelto 4 — Normalizar una tabla desnormalizada

In [ ]:
import sqlite3

# ✅ Normalización paso a paso

# ─── ANTES: tabla desnormalizada ───
print('=== TABLA DESNORMALIZADA ===')
datos_originales = [
    # pedido_id, cliente, email,         ciudad,       producto,  cat,           precio
    (1, 'Ana García', 'ana@mail.com', 'Buenos Aires', 'Laptop Pro',   'Electrónica', 1200.00),
    (1, 'Ana García', 'ana@mail.com', 'Buenos Aires', 'Mouse USB',    'Electrónica',   25.00),
    (2, 'Ana García', 'ana@mail.com', 'Buenos Aires', 'Auriculares',  'Electrónica',   89.99),
    (3, 'Luis Pérez', 'luis@mail.com','Córdoba',       'Camiseta',     'Ropa',          25.00),
    (3, 'Luis Pérez', 'luis@mail.com','Córdoba',       'Pantalón',     'Ropa',          60.00),
]
print(f'  {'pedido':8} {'cliente':12} {'email':18} {'ciudad':15} {'producto':15} {'categoria':14} {'precio':>8}')
print('  ' + '-'*95)
for row in datos_originales:
    print(f'  {row[0]:<8} {row[1]:12} {row[2]:18} {row[3]:15} {row[4]:15} {row[5]:14} {row[6]:>8.2f}')

print('\n⚠️  Problemas:')
print('  - Ana aparece 3 veces → si cambia su ciudad hay que actualizar 3 filas')
print('  - Electrónica aparece 3 veces → redundancia')
print('  - Si borramos el pedido 3 perdemos a Luis')

# ─── DESPUÉS: normalizada (3NF) ───
print('\n=== DISEÑO NORMALIZADO (3NF) ===')
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()
cursor.executescript("""
    CREATE TABLE categorias  (id INTEGER PRIMARY KEY, nombre TEXT UNIQUE);
    CREATE TABLE clientes    (id INTEGER PRIMARY KEY, nombre TEXT, email TEXT UNIQUE, ciudad TEXT);
    CREATE TABLE productos   (id INTEGER PRIMARY KEY, nombre TEXT, precio REAL, cat_id INTEGER REFERENCES categorias(id));
    CREATE TABLE pedidos     (id INTEGER PRIMARY KEY, cliente_id INTEGER REFERENCES clientes(id));
    CREATE TABLE items_pedido(pedido_id INTEGER REFERENCES pedidos(id), producto_id INTEGER REFERENCES productos(id), PRIMARY KEY(pedido_id, producto_id));
""")
cursor.executemany('INSERT INTO categorias VALUES (?, ?)', [(1,'Electrónica'),(2,'Ropa')])
cursor.executemany('INSERT INTO clientes VALUES (?, ?, ?, ?)', [(1,'Ana García','ana@mail.com','Buenos Aires'),(2,'Luis Pérez','luis@mail.com','Córdoba')])
cursor.executemany('INSERT INTO productos VALUES (?, ?, ?, ?)', [(1,'Laptop Pro',1200.00,1),(2,'Mouse USB',25.00,1),(3,'Auriculares',89.99,1),(4,'Camiseta',25.00,2),(5,'Pantalón',60.00,2)])
cursor.executemany('INSERT INTO pedidos VALUES (?, ?)', [(1,1),(2,1),(3,2)])
cursor.executemany('INSERT INTO items_pedido VALUES (?, ?)', [(1,1),(1,2),(2,3),(3,4),(3,5)])
conn.commit()

print('  5 tablas separadas — cada dato está en UN solo lugar')
print('  Si Ana cambia de ciudad: 1 sola actualización en tabla clientes')
print('  Si una categoría cambia de nombre: 1 sola actualización en tabla categorias')

# Reconstruir la vista original con JOIN
print('\n=== MISMA INFO con JOIN (sin redundancia) ===')
for row in cursor.execute("""
    SELECT p.id, c.nombre, c.email, c.ciudad, pr.nombre, cat.nombre, pr.precio
    FROM items_pedido ip
    JOIN pedidos  p   ON ip.pedido_id   = p.id
    JOIN clientes c   ON p.cliente_id   = c.id
    JOIN productos pr ON ip.producto_id = pr.id
    JOIN categorias cat ON pr.cat_id    = cat.id
    ORDER BY p.id
"""):
    print(f'  Pedido {row[0]}: {row[1]} → {row[4]} (${row[6]:.2f})')
conn.close()

### ✏️ Ejercicio guiado 4 — Identificar y corregir violaciones

Analizá la siguiente tabla desnormalizada e identificá qué forma normal viola. Luego implementá la versión normalizada.

```
TABLA: inscripciones_curso
┌──────────┬─────────────┬───────────────┬──────────────┬───────────────┬──────────────┐
│ alumno_id│ alumno_nombre│ alumno_email  │ curso_id     │ curso_nombre  │ instructor   │
├──────────┼─────────────┼───────────────┼──────────────┼───────────────┼──────────────┤
│ 1        │ María        │ m@mail.com    │ 101          │ Python Básico │ Prof. López  │
│ 1        │ María        │ m@mail.com    │ 102          │ SQL Avanzado  │ Prof. García │
│ 2        │ Juan         │ j@mail.com    │ 101          │ Python Básico │ Prof. López  │
└──────────┴─────────────┴───────────────┴──────────────┴───────────────┴──────────────┘
```

**Pistas:**
- ¿Qué pasa si Prof. López cambia de nombre?
- ¿Cuántas actualizaciones necesitaría?
- Necesitás al menos 3 tablas: `alumnos`, `cursos`, `inscripciones`

In [ ]:
import sqlite3

# ✏️ Paso 1: identificá las violaciones de normalización
# Escribí tu análisis aquí como comentario:
# Viola: ...
# Porque: ...

# ✏️ Paso 2: implementá la versión normalizada
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Crear tablas normalizadas:


# Insertar datos equivalentes:


# Verificá con un JOIN que recupera la misma info que la tabla original:


conn.close()


<details>
<summary>👀 Ver solución</summary>

```python
# Violaciones:
# - 2NF: alumno_nombre y alumno_email dependen solo de alumno_id (no del curso)
# - 2NF: curso_nombre e instructor dependen solo de curso_id (no del alumno)
# - Si Prof. López cambia de nombre → hay que actualizar todas sus filas

cursor.executescript("""
    CREATE TABLE alumnos (id INTEGER PRIMARY KEY, nombre TEXT, email TEXT UNIQUE);
    CREATE TABLE cursos  (id INTEGER PRIMARY KEY, nombre TEXT, instructor TEXT);
    CREATE TABLE inscripciones (
        alumno_id INTEGER REFERENCES alumnos(id),
        curso_id  INTEGER REFERENCES cursos(id),
        PRIMARY KEY (alumno_id, curso_id)
    );
""")
cursor.executemany('INSERT INTO alumnos VALUES (?,?,?)', [(1,'María','m@mail.com'),(2,'Juan','j@mail.com')])
cursor.executemany('INSERT INTO cursos VALUES (?,?,?)', [(101,'Python Básico','Prof. López'),(102,'SQL Avanzado','Prof. García')])
cursor.executemany('INSERT INTO inscripciones VALUES (?,?)', [(1,101),(1,102),(2,101)])
```
</details>

### 🚀 Ejercicio libre 4 — Normalización completa

La siguiente tabla está completamente desnormalizada. Llevala hasta **3NF**:

```
TABLA: hospital_desnormalizado
paciente_id | paciente_nombre | medico_id | medico_nombre | especialidad | fecha_turno | diagnostico | medicamento | dosis
    1        | Carlos         |    10     | Dr. Ramos     | Cardiología  | 2024-01-05  | Hipertensión| Enalapril   | 10mg
    1        | Carlos         |    10     | Dr. Ramos     | Cardiología  | 2024-01-05  | Hipertensión| Aspirina    | 100mg
    2        | Sofía          |    11     | Dra. Vega     | Neurología   | 2024-01-10  | Migraña     | Ibuprofeno  | 400mg
```

1. Identificá todas las dependencias y redundancias
2. Diseñá las tablas normalizadas
3. Implementá y verificá con JOINs

In [ ]:
import sqlite3

# 🚀 Tu análisis y solución aquí:
# Entidades identificadas:
# Relaciones:



---
## ✏️ Bloque 5: Optimización — Reescritura de Consultas y CTEs

### 📘 Teoría

Una consulta compleja puede reescribirse de formas más eficientes y legibles.

#### Subconsultas (Subqueries)
Una consulta dentro de otra. Útil pero puede ser lenta si retorna muchas filas:
```sql
-- Clientes que gastaron más que el promedio
SELECT nombre FROM clientes
WHERE id IN (
    SELECT cliente_id FROM pedidos
    WHERE total > (SELECT AVG(total) FROM pedidos)
);
```

#### CTEs — Common Table Expressions
Con `WITH` definís una consulta temporal con nombre. Más legible y reutilizable:
```sql
WITH ventas_por_cliente AS (
    SELECT cliente_id, SUM(total) as total_gastado
    FROM pedidos
    GROUP BY cliente_id
),
promedio AS (
    SELECT AVG(total_gastado) as promedio FROM ventas_por_cliente
)
SELECT c.nombre, vpc.total_gastado
FROM ventas_por_cliente vpc
JOIN clientes c ON vpc.cliente_id = c.id
JOIN promedio  p ON vpc.total_gastado > p.promedio;
```

#### Reglas generales de optimización
1. Filtrá temprano con `WHERE` — reducís filas antes del JOIN
2. Seleccioná solo las columnas que necesitás — evitá `SELECT *` en producción
3. Usá `EXISTS` en vez de `IN` para subconsultas grandes
4. Verificá con `EXPLAIN QUERY PLAN` que se usen índices

### 💡 Ejemplo resuelto 5 — Consulta compleja con CTE

In [ ]:
import sqlite3

# ✅ Reescribir una consulta compleja usando CTEs

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()
cursor.executescript("""
    CREATE TABLE clientes (id INTEGER PRIMARY KEY, nombre TEXT, ciudad TEXT);
    CREATE TABLE productos (id INTEGER PRIMARY KEY, nombre TEXT, categoria TEXT, precio REAL);
    CREATE TABLE pedidos (id INTEGER PRIMARY KEY, cliente_id INTEGER, fecha TEXT, estado TEXT);
    CREATE TABLE detalle (pedido_id INTEGER, producto_id INTEGER, cantidad INTEGER, precio_unit REAL);
""")
cursor.executemany('INSERT INTO clientes VALUES (?,?,?)', [(1,'Ana','Buenos Aires'),(2,'Luis','Córdoba'),(3,'Marta','Rosario'),(4,'Pedro','Mendoza')])
cursor.executemany('INSERT INTO productos VALUES (?,?,?,?)', [(1,'Laptop','Electrónica',1200),(2,'Mouse','Electrónica',25),(3,'Camiseta','Ropa',25),(4,'Libro','Educación',30),(5,'Sillón','Hogar',350)])
cursor.executemany('INSERT INTO pedidos VALUES (?,?,?,?)', [(1,1,'2024-01','entregado'),(2,1,'2024-02','enviado'),(3,2,'2024-01','entregado'),(4,3,'2024-02','pendiente')])
cursor.executemany('INSERT INTO detalle VALUES (?,?,?,?)', [(1,1,1,1200),(1,2,2,25),(2,3,3,25),(3,4,2,30),(3,5,1,350),(4,1,1,1200)])
conn.commit()

# VERSIÓN BÁSICA (difícil de leer, subconsulta anidada)
print('=== Versión básica (subconsultas anidadas) ===')
query_basica = """
    SELECT nombre FROM clientes WHERE id IN (
        SELECT cliente_id FROM pedidos WHERE id IN (
            SELECT pedido_id FROM detalle
            GROUP BY pedido_id
            HAVING SUM(cantidad * precio_unit) > 500
        )
    )
"""
print('  Clientes con algún pedido > $500:', [r[0] for r in cursor.execute(query_basica)])

# VERSIÓN CON CTEs (más clara y mantenible)
print('\n=== Versión con CTEs (más legible) ===')
query_cte = """
    WITH total_por_pedido AS (
        SELECT pedido_id, SUM(cantidad * precio_unit) AS total
        FROM detalle
        GROUP BY pedido_id
    ),
    pedidos_grandes AS (
        SELECT p.cliente_id, tp.total
        FROM pedidos p
        JOIN total_por_pedido tp ON p.id = tp.pedido_id
        WHERE tp.total > 500
    )
    SELECT DISTINCT c.nombre, pg.total
    FROM clientes c
    JOIN pedidos_grandes pg ON c.id = pg.cliente_id
    ORDER BY pg.total DESC
"""
print('  Clientes con algún pedido > $500:')
for row in cursor.execute(query_cte):
    print(f'    {row[0]:12} | ${row[1]:.2f}')

# Análisis por ciudad usando CTE
print('\n=== Gasto total por ciudad (CTE encadenadas) ===')
query_ciudad = """
    WITH gasto_cliente AS (
        SELECT p.cliente_id, SUM(d.cantidad * d.precio_unit) as total
        FROM pedidos p JOIN detalle d ON p.id = d.pedido_id
        GROUP BY p.cliente_id
    )
    SELECT c.ciudad, COUNT(DISTINCT c.id) as clientes, ROUND(SUM(gc.total),2) as gasto_total
    FROM clientes c
    JOIN gasto_cliente gc ON c.id = gc.cliente_id
    GROUP BY c.ciudad
    ORDER BY gasto_total DESC
"""
for row in cursor.execute(query_ciudad):
    print(f'  {row[0]:15} | {row[1]} cliente(s) | ${row[2]:.2f}')

conn.close()

### ✏️ Ejercicio guiado 5 — Reescribir con CTEs

Tomá la siguiente consulta anidada y reescribila usando CTEs. Luego verificá que ambas dan el mismo resultado.

**Consulta original:**
```sql
SELECT nombre FROM empleados
WHERE depto_id IN (
    SELECT id FROM departamentos
    WHERE presupuesto > (
        SELECT AVG(presupuesto) FROM departamentos
    )
);
```

In [ ]:
import sqlite3

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()
cursor.executescript("""
    CREATE TABLE departamentos (id INTEGER PRIMARY KEY, nombre TEXT, presupuesto REAL);
    CREATE TABLE empleados (id INTEGER PRIMARY KEY, nombre TEXT, depto_id INTEGER, salario REAL);
""")
cursor.executemany('INSERT INTO departamentos VALUES (?,?,?)', [(1,'IT',500000),(2,'Ventas',300000),(3,'RRHH',150000),(4,'Marketing',400000)])
cursor.executemany('INSERT INTO empleados VALUES (?,?,?,?)', [(1,'Ana',1,85000),(2,'Luis',2,60000),(3,'Marta',1,90000),(4,'Carlos',3,55000),(5,'Laura',4,75000),(6,'Pedro',2,58000)])
conn.commit()

# Versión original (subconsultas anidadas)
query_original = """
    SELECT nombre FROM empleados
    WHERE depto_id IN (
        SELECT id FROM departamentos
        WHERE presupuesto > (SELECT AVG(presupuesto) FROM departamentos)
    )
"""
resultado_original = [r[0] for r in cursor.execute(query_original)]
print('Versión original:', resultado_original)

# ✏️ Tu versión con CTEs aquí:
query_cte = """
    WITH
    -- Paso 1: calcular el promedio de presupuesto
    -- Paso 2: filtrar departamentos con presupuesto > promedio
    -- Paso 3: obtener empleados de esos departamentos
    SELECT 'completar' as nombre  -- reemplazá esto
"""
resultado_cte = [r[0] for r in cursor.execute(query_cte)]
print('Versión CTE:    ', resultado_cte)
print('¿Mismos resultados?', sorted(resultado_original) == sorted(resultado_cte))

conn.close()

<details>
<summary>👀 Ver solución</summary>

```python
query_cte = """
    WITH
    promedio_presupuesto AS (
        SELECT AVG(presupuesto) as prom FROM departamentos
    ),
    deptos_grandes AS (
        SELECT d.id
        FROM departamentos d, promedio_presupuesto p
        WHERE d.presupuesto > p.prom
    )
    SELECT e.nombre
    FROM empleados e
    JOIN deptos_grandes dg ON e.depto_id = dg.id
"""
# Resultado: Ana, Marta, Laura (IT y Marketing tienen presupuesto > promedio de 337.500)
```
</details>

### 🚀 Ejercicio libre 5 — Análisis completo de ventas

Usando la BD del ejercicio libre 1 (sistema de ventas), respondé estas preguntas **usando CTEs**:

1. ¿Cuál es el **ticket promedio** por cliente? ¿Quiénes están por encima del promedio?
2. ¿Cuál es el **producto más vendido por categoría**?
3. ¿Cuántos clientes compraron **más de una vez**? ¿Quiénes son?
4. Creá un **ranking de clientes** por total gastado con su posición (1°, 2°, 3°...)

> 💡 Para el ranking, SQLite soporta funciones de ventana: `RANK() OVER (ORDER BY total DESC)`

In [ ]:
import sqlite3

# 🚀 Tu solución aquí:


---
## ✅ Resumen de la Semana 3-4

### Lo que aprendiste

| Tema | Conceptos clave |
|------|-----------------|
| JOINs | `INNER`, `LEFT`, `RIGHT` — cómo combinar tablas |
| Agregación | `COUNT`, `SUM`, `AVG`, `GROUP BY`, `HAVING` |
| Índices | Crear, cuándo usarlos, `EXPLAIN QUERY PLAN` |
| Modelado ER | Entidades, relaciones 1:1, 1:N, N:M, tablas intermedias |
| Normalización | 1NF, 2NF, 3NF — eliminar redundancias |
| CTEs | `WITH ... AS` para queries más legibles y mantenibles |

### ➡️ Próximos pasos — Semana 5-6
- Arquitectura cliente-servidor y comunicación HTTP
- APIs REST con Python (`requests`, Flask)
- Manipulación avanzada de datos con `pandas` y `NumPy`

---

### 📚 Recursos recomendados
- [SQL JOIN Visualizer](https://joins.spathon.com/) — visual interactivo de los tipos de JOIN
- [SQLZoo — SQL Tutorial](https://sqlzoo.net/) — práctica online
- [Use The Index, Luke!](https://use-the-index-luke.com/) — guía profunda sobre índices
- [Database Normalization](https://www.databasestar.com/database-normalization/) — guía completa